In [3]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS
import statsmodels.api as sm

PATHS = {
    'M0': '../output/results/M0_results.csv',
    'M1': '../output/results/M1_results.csv',
    'M2': '../output/results/M2_results.csv',
}

DATE_COL = 'date'
COUNTRY_COL = 'country'
DD_COL = 'distance_to_distress'
CDS_COL = 'cds_spread'
HORIZONS = [1, 2, 4, 8]

EXPORTERS = ['Saudi Arabia', 'Abu Dhabi', 'Qatar', 'Colombia', 'Mexico', 'Brazil', 'Egypt', 'Malaysia']
CONTROLS = ['Chile', 'China', 'Indonesia', 'Philippines', 'South Africa', 'South Korea', 'Thailand', 'Turkey']

In [6]:
import pandas as pd
import statsmodels.api as sm

results = []

for model_name, path in PATHS.items():
    df = pd.read_csv(path, parse_dates=[DATE_COL])
    df = df.sort_values([COUNTRY_COL, DATE_COL])
    df['delta_dd'] = df.groupby(COUNTRY_COL)[DD_COL].diff(1)
    df['delta_cds'] = df.groupby(COUNTRY_COL)[CDS_COL].diff(1)
    df = df.dropna(subset=['delta_dd', 'delta_cds'])

    for country in df[COUNTRY_COL].unique():
        cdf = df[df[COUNTRY_COL] == country]
        X = sm.add_constant(cdf['delta_dd'])
        y = cdf['delta_cds']
        ols = sm.OLS(y, X).fit(cov_type='HAC', cov_kwds={'maxlags': 1})

        results.append({
            'model':   model_name,
            'country': country,
            'alpha':   ols.params['const'],
            'beta':    ols.params['delta_dd'],
            'se_beta': ols.bse['delta_dd'],
            'p_value': ols.pvalues['delta_dd'],
            'r2':      ols.rsquared,
            'n_obs':   int(ols.nobs),
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

model      country     alpha       beta  se_beta      p_value           r2  n_obs
   M0    Abu Dhabi -0.048454   0.009284 0.019735 6.380360e-01 6.172783e-05    521
   M0       Brazil  0.080510 -11.114221 3.971007 5.128622e-03 4.359271e-02    521
   M0        Chile -0.065489  -0.996381 0.599423 9.646573e-02 3.493765e-03    521
   M0        China -0.044083   0.003157 0.019568 8.718361e-01 2.227350e-05    521
   M0     Colombia  0.112138  -3.698080 2.031846 6.874996e-02 1.444087e-02    521
   M0        Dubai -0.316817  -0.049744 0.042500 2.418207e-01 4.755885e-04    521
   M0        Egypt  0.529919   0.093451 0.409869 8.196444e-01 9.896945e-06    521
   M0    Indonesia -0.161698  -0.076439 0.169239 6.515111e-01 1.200647e-04    521
   M0     Malaysia -0.119059  -0.084139 0.057585 1.439792e-01 5.418693e-04    521
   M0       Mexico -0.004010  -6.801668 3.280792 3.815565e-02 7.919980e-02    521
   M0  Philippines -0.054434  -0.001768 0.062165 9.773069e-01 2.206634e-07    521
   M0        Qat

In [15]:
import pandas as pd
import statsmodels.api as sm

def format_p_value(p):
    """Formats p-value: 2 decimals + significance stars/symbols."""
    if pd.isna(p):
        return ""
    
    # Significance markers
    # * for 5% (p <= 0.05)
    # + for 10% (0.05 < p <= 0.10)
    if p < 0.001:
        sig = "***"
    elif p < 0.01:
        sig = "**"
    elif p <= 0.05:
        sig = "*"
    elif p <= 0.10:
        sig = "+"
    else:
        sig = ""
        
    return f"{p:.2f}{sig}"

def j_test(y, X1, X2, hac_lags=20):
    """
    Test M1 against M2.
    Step 1: fit M2, get fitted values
    Step 2: add fitted values to M1, test their significance
    """
    # Fit M2, extract fitted values
    fit2 = sm.OLS(y, X2).fit()
    yhat2 = fit2.fittedvalues

    # Add M2 fitted values to M1
    X1_aug = X1.copy()
    X1_aug['yhat_alt'] = yhat2.values
    fit1_aug = sm.OLS(y, X1_aug).fit(cov_type='HAC', cov_kwds={'maxlags': hac_lags})

    return {
        'coef_yhat': fit1_aug.params['yhat_alt'],
        'p_value':   fit1_aug.pvalues['yhat_alt'],
    }

results = []

for country in EXPORTERS + CONTROLS:
    dfs = {}
    for model_name, path in PATHS.items():
        df = pd.read_csv(path, parse_dates=[DATE_COL])
        df = df.sort_values([COUNTRY_COL, DATE_COL])
        df['delta_dd'] = df.groupby(COUNTRY_COL)[DD_COL].diff(1)
        df['delta_cds'] = df.groupby(COUNTRY_COL)[CDS_COL].diff(1)
        cdf = df[df[COUNTRY_COL] == country].dropna(subset=['delta_dd', 'delta_cds'])
        dfs[model_name] = cdf.set_index(DATE_COL)

    common_dates = dfs['M0'].index\
        .intersection(dfs['M1'].index)\
        .intersection(dfs['M2'].index)

    if len(common_dates) == 0:
        print(f"No common dates for {country}, skipping")
        continue

    y    = dfs['M0'].loc[common_dates, 'delta_cds']
    X_M0 = sm.add_constant(dfs['M0'].loc[common_dates, 'delta_dd'].rename('dd_M0'))
    X_M1 = sm.add_constant(dfs['M1'].loc[common_dates, 'delta_dd'].rename('dd_M1'))
    X_M2 = sm.add_constant(dfs['M2'].loc[common_dates, 'delta_dd'].rename('dd_M2'))

    # Run J-tests
    j_m0_given_m1 = j_test(y, X_M0, X_M1)
    j_m1_given_m0 = j_test(y, X_M1, X_M0)
    j_m0_given_m2 = j_test(y, X_M0, X_M2)
    j_m2_given_m0 = j_test(y, X_M2, X_M0)

    results.append({
        'country':         country,
        'group':           'Exporter' if country in EXPORTERS else 'Control',
        'M1_adds_to_M0_p': j_m0_given_m1['p_value'],
        'M0_adds_to_M1_p': j_m1_given_m0['p_value'],
        'M2_adds_to_M0_p': j_m0_given_m2['p_value'],
        'M0_adds_to_M2_p': j_m2_given_m0['p_value'],
    })

# Create DataFrame
results_df = pd.DataFrame(results)

# Apply the formatting to p-value columns
p_cols = ['M1_adds_to_M0_p', 'M0_adds_to_M1_p', 'M2_adds_to_M0_p', 'M0_adds_to_M2_p']
for col in p_cols:
    results_df[col] = results_df[col].apply(format_p_value)

print(results_df.to_string(index=False))

     country    group M1_adds_to_M0_p M0_adds_to_M1_p M2_adds_to_M0_p M0_adds_to_M2_p
Saudi Arabia Exporter            0.21            0.22           0.05+            0.32
   Abu Dhabi Exporter            0.18            0.19         0.00***           0.05+
       Qatar Exporter            0.21            0.14         0.00***            0.16
    Colombia Exporter           0.05*            0.28          0.00**            0.14
      Mexico Exporter          0.01**            0.22          0.00**            0.29
      Brazil Exporter          0.00**            0.18            0.28            0.96
       Egypt Exporter            0.70            0.66            0.35            0.27
    Malaysia Exporter           0.06+           0.08+         0.00***            0.53
       Chile  Control          0.01**           0.05+           0.04*            0.10
       China  Control          0.00**          0.00**            0.17            0.29
   Indonesia  Control            0.16            0.18 

In [30]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS

HORIZON = 1
results = []

for model_name, path in PATHS.items():
    df = pd.read_csv(path, parse_dates=[DATE_COL])
    df = df.sort_values([COUNTRY_COL, DATE_COL])
    df['delta_dd']  = df.groupby(COUNTRY_COL)[DD_COL].diff(HORIZON)
    df['delta_cds'] = df.groupby(COUNTRY_COL)[CDS_COL].diff(HORIZON)
    df = df.dropna(subset=['delta_dd', 'delta_cds'])

    ctrl_reset = controls_weekly.reset_index().rename(columns={'date': DATE_COL})
    df = pd.merge_asof(
        df.sort_values(DATE_COL),
        ctrl_reset.sort_values(DATE_COL),
        on=DATE_COL,
        direction='nearest',
        tolerance=pd.Timedelta('7 days')
    )
    df = df.dropna(subset=['VIX_chg', 'DXY_chg', 'UST10Y_chg'])

    for group_name, group_countries in [('Exporters', EXPORTERS), ('Controls', CONTROLS)]:
        gdf = df[df[COUNTRY_COL].isin(group_countries)].dropna(
            subset=['delta_dd', 'delta_cds', 'VIX_chg', 'DXY_chg', 'UST10Y_chg']
        ).copy()

        if len(gdf) == 0:
            print(f"Empty: {model_name} {group_name}")
            continue

        # Set panel index
        gdf = gdf.set_index([COUNTRY_COL, DATE_COL])
        y = gdf['delta_cds']

        # Without global controls, with country FE
        X_base = gdf[['delta_dd']]
        res_base = PanelOLS(y, X_base, entity_effects=True).fit(
            cov_type='clustered', cluster_entity=True
        )

        # With global controls, with country FE
        X_ctrl = gdf[['delta_dd', 'VIX_chg', 'DXY_chg', 'UST10Y_chg']]
        res_ctrl = PanelOLS(y, X_ctrl, entity_effects=True).fit(
            cov_type='clustered', cluster_entity=True
        )

        results.append({
            'model':     model_name,
            'group':     group_name,
            'beta_base': res_base.params['delta_dd'],
            'p_base':    res_base.pvalues['delta_dd'],
            'beta_ctrl': res_ctrl.params['delta_dd'],
            'p_ctrl':    res_ctrl.pvalues['delta_dd'],
            'n_obs':     int(res_ctrl.nobs),
            'n_groups':  int(res_ctrl.entity_info.total),
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

model     group  beta_base   p_base  beta_ctrl   p_ctrl  n_obs  n_groups
   M0 Exporters  -0.046822 0.299934  -0.057338 0.196610   4168         8
   M0  Controls  -0.035383 0.376823  -0.035376 0.356191   4168         8
   M1 Exporters  -2.985765 0.008809  -2.965158 0.006543   4168         8
   M1  Controls  -1.882470 0.034952  -1.806767 0.034420   4168         8
   M2 Exporters  -1.162914 0.000525  -1.059640 0.000474   4168         8
   M2  Controls  -0.518814 0.040652  -0.455542 0.047648   4168         8


In [35]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS

HORIZON = 2
interaction_results = []

for model_name, path in PATHS.items():
    df = pd.read_csv(path, parse_dates=[DATE_COL])
    df = df.sort_values([COUNTRY_COL, DATE_COL])
    df['delta_dd']  = df.groupby(COUNTRY_COL)[DD_COL].diff(HORIZON)
    df['delta_cds'] = df.groupby(COUNTRY_COL)[CDS_COL].diff(HORIZON)
    df = df.dropna(subset=['delta_dd', 'delta_cds'])

    ctrl_reset = controls_weekly.reset_index().rename(columns={'date': DATE_COL})
    df = pd.merge_asof(
        df.sort_values(DATE_COL),
        ctrl_reset.sort_values(DATE_COL),
        on=DATE_COL,
        direction='nearest',
        tolerance=pd.Timedelta('7 days')
    )
    df = df.dropna(subset=['VIX_chg', 'DXY_chg', 'UST10Y_chg'])

    gdf = df[df[COUNTRY_COL].isin(EXPORTERS + CONTROLS)].copy()
    gdf['exporter'] = gdf[COUNTRY_COL].isin(EXPORTERS).astype(float)
    gdf['dd_x_exp'] = gdf['delta_dd'] * gdf['exporter']

    gdf = gdf.set_index([COUNTRY_COL, DATE_COL])
    y   = gdf['delta_cds']

    # Without global controls, cluster by time
    X_base = gdf[['delta_dd', 'dd_x_exp']]
    res_base = PanelOLS(y, X_base, entity_effects=True).fit(
        cov_type='clustered', cluster_time=True
    )

    # With global controls, cluster by time
    X_ctrl = gdf[['delta_dd', 'dd_x_exp', 'VIX_chg', 'DXY_chg', 'UST10Y_chg']]
    res_ctrl = PanelOLS(y, X_ctrl, entity_effects=True).fit(
        cov_type='clustered', cluster_time=True
    )

    interaction_results.append({
        'model':         model_name,
        'beta_dd':       res_ctrl.params['delta_dd'],
        'p_dd':          res_ctrl.pvalues['delta_dd'],
        'beta_interact': res_ctrl.params['dd_x_exp'],
        'p_interact':    res_ctrl.pvalues['dd_x_exp'],
        'beta_dd_base':  res_base.params['delta_dd'],
        'p_dd_base':     res_base.pvalues['delta_dd'],
        'beta_int_base': res_base.params['dd_x_exp'],
        'p_int_base':    res_base.pvalues['dd_x_exp'],
        'n_obs':         int(res_ctrl.nobs),
    })

int_df = pd.DataFrame(interaction_results)
print(int_df.to_string(index=False))

model   beta_dd     p_dd  beta_interact  p_interact  beta_dd_base  p_dd_base  beta_int_base  p_int_base  n_obs
   M0 -0.099284 0.016226       0.012035    0.880065     -0.100845   0.020408       0.013491    0.869263   8320
   M1 -2.279993 0.003944      -0.738011    0.669258     -2.454626   0.005232      -0.735377    0.674151   8320
   M2 -0.751433 0.000011      -0.521748    0.156236     -0.847705   0.000003      -0.600174    0.109393   8320
